# 基于 HMM 的中文词性标注系统

## 完整流程
1. **语料加载与预处理** — 合并 PKU98 + PKU-SEGPOS 两个数据源
2. **HMM 三要素统计** — 从训练集中统计 π（初始概率）、A（转移概率）、B（观测概率）
3. **Viterbi 解码** — 输入分词后的句子，输出最可能的词性序列
4. **模型评估** — 在测试集上计算标注准确率

In [1]:
import os
import sys
import json
import math
import random
from collections import defaultdict

BASE_DIR = os.path.dirname(os.path.abspath('__file__'))
sys.path.insert(0, os.path.join(BASE_DIR, 'src'))

from corpus_loader import load_pku98_corpus, load_pku_segpos_jsonl, corpus_statistics
from hmm_pos_tagger import compute_hmm_statistics, save_model
from viterbi_decoder import ViterbiDecoder

print('模块加载完成')

模块加载完成


---
## 第一步：语料加载与预处理

加载两个数据源并合并为训练集和测试集。

In [2]:
# 加载 PKU98（1998年1月人民日报）
pku98_train = load_pku98_corpus('data/raw/pku98/pku98/199801-train.txt')
pku98_test = load_pku98_corpus('data/raw/pku98/pku98/199801-test.txt')
print(f'PKU98: 训练 {len(pku98_train)} 句, 测试 {len(pku98_test)} 句')

# 加载 PKU-SEGPOS（2000年人民日报）
segpos_dir = 'data/raw/pku_segpos/PKU-SEGPOS/extracted'
segpos_train = load_pku_segpos_jsonl(os.path.join(segpos_dir, 'train.jsonline'))
segpos_dev = load_pku_segpos_jsonl(os.path.join(segpos_dir, 'dev.jsonline'))
print(f'PKU-SEGPOS: 训练 {len(segpos_train)} 句, 开发 {len(segpos_dev)} 句')

# 合并
all_train = pku98_train + segpos_train
all_test = pku98_test + segpos_dev
random.seed(42)
random.shuffle(all_train)

print(f'\n合并后: 训练 {len(all_train)} 句, 测试 {len(all_test)} 句')

PKU98: 训练 17524 句, 测试 1949 句


PKU-SEGPOS: 训练 31772 句, 开发 5204 句

合并后: 训练 49296 句, 测试 7153 句


In [3]:
# 查看语料统计
train_stats = corpus_statistics(all_train)
print(f"训练集: {train_stats['num_sentences']} 句, {train_stats['total_words']} 词, "
      f"{train_stats['vocab_size']} 词汇, {train_stats['num_tags']} 种标签")

test_stats = corpus_statistics(all_test)
print(f"测试集: {test_stats['num_sentences']} 句, {test_stats['total_words']} 词")

训练集: 49296 句, 2578800 词, 157757 词汇, 57 种标签
测试集: 7153 句, 381759 词


In [4]:
# 查看示例句子
print('训练集示例句子:')
for i, sent in enumerate(all_train[:3]):
    tagged = ' '.join(f'{w}/{t}' for w, t in sent[:10])
    print(f'  [{i+1}] {tagged} ...')

训练集示例句子:
  [1] 很多/m “/w 黑客/n ”/w 可以/v 在/p 互联网/n 上/f 秘密/ad 破解/v ...
  [2] 近年来/l 这/r 拜年/v 也/d 打/v 上/v 了/u 时代/n 烙印/n 听说/v ...
  [3] 刘/nr 长春/nr ...


---
## 第二步：统计 HMM 三要素

从训练语料中统计三个核心概率：

| 要素 | 含义 | 公式 |
|------|------|------|
| **π（初始概率）** | 句首为某标签的概率 | P(tag) = start_count(tag) / total_sentences |
| **A（转移概率）** | 标签 i 后接标签 j 的概率 | P(tag_j \| tag_i) = trans_count(tag_i, tag_j) / count(tag_i) |
| **B（观测概率）** | 标签下出现某词语的概率 | P(word \| tag) = emit_count(tag, word) / count(tag) |

In [5]:
# 统计 HMM 三要素（含拉普拉斯平滑）
stats = compute_hmm_statistics(all_train)

第一步：从语料中统计 HMM 三要素

语料规模: 49296 句


总词数: 2578800
标签数: 57
词表大小: 157757

计算概率（拉普拉斯平滑）...



------------------------------------------------------------
统计结果
------------------------------------------------------------

【π 初始状态概率】Top-10:
  n     : P=0.123782  (句首出现 6108 次)
  r     : P=0.116386  (句首出现 5743 次)
  nr    : P=0.108301  (句首出现 5344 次)
  p     : P=0.105100  (句首出现 5186 次)
  v     : P=0.099467  (句首出现 4908 次)
  ns    : P=0.098576  (句首出现 4864 次)
  t     : P=0.071404  (句首出现 3523 次)
  m     : P=0.056835  (句首出现 2804 次)
  nt    : P=0.044800  (句首出现 2210 次)
  w     : P=0.044030  (句首出现 2172 次)

【A 状态转移概率】示例:
  v    → n(0.212), u(0.145), w(0.117), v(0.076), m(0.067)
  n    → w(0.196), v(0.189), u(0.102), n(0.084), d(0.067)
  d    → v(0.658), a(0.138), p(0.079), d(0.032), m(0.015)
  p    → n(0.288), v(0.140), r(0.103), ns(0.102), t(0.069)

【B 观测概率】示例:
  n   : 人(0.0100), 企业(0.0079), 记者(0.0078), 经济(0.0068), 问题(0.0067)
  v   : 是(0.0376), 有(0.0169), 说(0.0089), 要(0.0084), 为(0.0078)
  a   : 新(0.0274), 大(0.0220), 好(0.0136), 重要(0.0117), 多(0.0108)


  nr  : 李(0.0076), 王(0.0058), 张(0.0043), 江(0.0042), 刘(0.0037)
  nt  : 新华社(0.0140), 公司(0.0050), 国务院(0.0048), 政府(0.0030), 联合国(0.0028)


In [6]:
# 保存模型参数
model_path = 'data/processed/hmm_model.json'
save_model(stats, model_path)


模型已保存: data/processed/hmm_model.json (301.9 MB)


---
## 第三步：Viterbi 解码

给定一个分词后的句子，使用 Viterbi 算法找出最可能的词性标签序列。

**算法核心**：动态规划
- δ[t][j] = 到时刻 t、状态为 tag_j 的最大 log 概率路径
- 递推：δ[t][j] = max_i(δ[t-1][i] + log A(j\|i)) + log B(word_t\|j)

In [7]:
# 加载训练好的模型
decoder = ViterbiDecoder(model_path)
print(f'模型加载完成: {len(decoder.tags)} 种标签')

模型加载完成: 57 种标签


In [8]:
# 测试几个句子
test_sentences = [
    ['我', '爱', '北京', '天安门'],
    ['中国', '经济', '持续', '增长'],
    ['江泽民', '主席', '发表', '新年', '讲话'],
    ['他', '在', '清华大学', '学习', '计算机', '科学'],
    ['国务院', '总理', '主持', '召开', '会议'],
]

print('Viterbi 解码结果:')
print('-' * 60)
for words in test_sentences:
    result = decoder.tag_sentence_str(words)
    print(f'  输入: {" ".join(words)}')
    print(f'  标注: {result}')
    print()

Viterbi 解码结果:
------------------------------------------------------------
  输入: 我 爱 北京 天安门
  标注: 我/r 爱/v 北京/ns 天安门/ns

  输入: 中国 经济 持续 增长
  标注: 中国/ns 经济/n 持续/v 增长/v

  输入: 江泽民 主席 发表 新年 讲话
  标注: 江泽民/nr 主席/n 发表/v 新年/t 讲话/n

  输入: 他 在 清华大学 学习 计算机 科学
  标注: 他/r 在/p 清华大学/nt 学习/v 计算机/n 科学/n

  输入: 国务院 总理 主持 召开 会议
  标注: 国务院/nt 总理/n 主持/v 召开/v 会议/n



---
## 第四步：模型评估

在测试集上计算词性标注准确率。

In [9]:
def evaluate(decoder, test_sentences):
    """在测试集上评估标注准确率"""
    total = 0
    correct = 0
    oov_total = 0
    oov_correct = 0

    for sent in test_sentences:
        words = [w for w, t in sent]
        gold_tags = [t for w, t in sent]
        pred_tags = decoder.decode(words)

        for word, gold, pred in zip(words, gold_tags, pred_tags):
            total += 1
            if gold == pred:
                correct += 1
            if word not in decoder.log_B.get(gold, {}):
                oov_total += 1
                if gold == pred:
                    oov_correct += 1

    accuracy = correct / total * 100 if total > 0 else 0
    oov_accuracy = oov_correct / oov_total * 100 if oov_total > 0 else 0

    return {
        'total': total,
        'correct': correct,
        'accuracy': accuracy,
        'oov_total': oov_total,
        'oov_correct': oov_correct,
        'oov_accuracy': oov_accuracy,
    }


results = evaluate(decoder, all_test)
print(f'测试集评估结果:')
print(f'  总词数:     {results["total"]}')
print(f'  正确标注:   {results["correct"]}')
print(f'  准确率:     {results["accuracy"]:.2f}%')
print(f'  未登录词:   {results["oov_total"]}')
print(f'  未登录词准确率: {results["oov_accuracy"]:.2f}%')

测试集评估结果:
  总词数:     381759
  正确标注:   338936
  准确率:     88.78%
  未登录词:   20294
  未登录词准确率: 44.63%


---
## 交互演示

输入任意分词后的句子，查看词性标注结果。

In [10]:
# 自定义输入
my_sentence = ['改革', '开放', '的', '政策', '推动', '了', '中国', '的', '发展']

tagged = decoder.tag_sentence(my_sentence)
print('输入词语:', ' '.join(w for w, t in tagged))
print('词性标注:', ' '.join(f'{w}/{t}' for w, t in tagged))
print()
print('标签说明:')
tag_desc = {'n': '名词', 'v': '动词', 'a': '形容词', 'd': '副词', 'p': '介词',
            'r': '代词', 'm': '数词', 'q': '量词', 'u': '助词', 'c': '连词',
            'nr': '人名', 'ns': '地名', 'nt': '机构名', 'nz': '其他专名',
            'vn': '名动词', 'w': '标点'}
for word, tag in tagged:
    desc = tag_desc.get(tag, tag)
    print(f'  {word:6s} -> {tag:4s} ({desc})')

输入词语: 改革 开放 的 政策 推动 了 中国 的 发展
词性标注: 改革/v 开放/v 的/u 政策/n 推动/v 了/u 中国/ns 的/u 发展/vn

标签说明:
  改革     -> v    (动词)
  开放     -> v    (动词)
  的      -> u    (助词)
  政策     -> n    (名词)
  推动     -> v    (动词)
  了      -> u    (助词)
  中国     -> ns   (地名)
  的      -> u    (助词)
  发展     -> vn   (名动词)


---
## 第五步：完整流水线 — 从原始句子到词性标注

前面的步骤需要手动提供分好词的列表。现在引入 jieba 分词，实现：

**原始句子 → jieba 分词 → HMM 词性标注 → 完整结果**

In [11]:
from segmenter import segment, segment_with_pos, segment_batch

# 封装一个端到端的标注函数
def tag_raw_sentence(sentence, decoder):
    """
    端到端词性标注：原始句子 → 分词 → HMM标注
    """
    words = segment(sentence)
    words = [w for w in words if w.strip()]
    return decoder.tag_sentence(words)


def tag_raw_sentence_str(sentence, decoder):
    """端到端标注，返回格式化字符串"""
    tagged = tag_raw_sentence(sentence, decoder)
    return ' '.join(f'{w}/{t}' for w, t in tagged)


print('分词模块加载完成')

分词模块加载完成


### 5.1 端到端测试：输入原始句子

In [12]:
# 测试：输入原始句子（不需要手动分词）
raw_sentences = [
    '我爱北京天安门',
    '中国国民经济持续快速增长',
    '江泽民主席在清华大学发表重要讲话',
    '改革开放的政策推动了中国经济的发展',
    '人工智能技术正在深刻改变人类的生活方式',
]

print('端到端标注结果（原始句子 → 分词 → HMM词性标注）:')
print('=' * 70)
for sent in raw_sentences:
    words = segment(sent)
    seg_str = ' / '.join(words)
    result = tag_raw_sentence_str(sent, decoder)
    print(f'原文: {sent}')
    print(f'分词: {seg_str}')
    print(f'标注: {result}')
    print()

Building prefix dict from the default dictionary ...


Loading model from cache C:\Users\ADMIN\AppData\Local\Temp\jieba.cache


端到端标注结果（原始句子 → 分词 → HMM词性标注）:


Loading model cost 0.531 seconds.


Prefix dict has been built successfully.


原文: 我爱北京天安门
分词: 我 / 爱 / 北京 / 天安门
标注: 我/r 爱/v 北京/ns 天安门/ns

原文: 中国国民经济持续快速增长
分词: 中国 / 国民经济 / 持续 / 快速增长
标注: 中国/ns 国民经济/n 持续/v 快速增长/n

原文: 江泽民主席在清华大学发表重要讲话
分词: 江泽民 / 主席 / 在 / 清华大学 / 发表 / 重要讲话
标注: 江泽民/nr 主席/n 在/p 清华大学/nt 发表/v 重要讲话/n

原文: 改革开放的政策推动了中国经济的发展
分词: 改革开放 / 的 / 政策 / 推动 / 了 / 中国 / 经济 / 的 / 发展
标注: 改革开放/v 的/u 政策/n 推动/v 了/u 中国/ns 经济/n 的/u 发展/vn

原文: 人工智能技术正在深刻改变人类的生活方式
分词: 人工智能 / 技术 / 正在 / 深刻 / 改变 / 人类 / 的 / 生活 / 方式
标注: 人工智能/ns 技术/n 正在/d 深刻/ad 改变/v 人类/n 的/u 生活/vn 方式/n



### 5.2 对比 jieba 自带词性标注 vs 我们的 HMM 标注

In [13]:
# 对比 jieba.posseg 和我们的 HMM 标注
test_sents = [
    '他正在研究自然语言处理技术',
    '北京大学是中国最著名的大学之一',
    '国务院总理主持召开了常务会议',
]

# 本轮标注中用到的词性标签含义
tag_desc = {
    'n': '名词', 'v': '动词', 'a': '形容词', 'd': '副词',
    'r': '代词', 't': '时间词', 'l': '习用语', 'u': '助词',
    'ns': '地名', 'nt': '机构名', 'vn': '名动词',
}

print('jieba 标注 vs HMM 标注对比:')
print('=' * 70)
print()
print('词性标签含义:')
for tag, desc in sorted(tag_desc.items(), key=lambda x: len(x[0])):
    print(f'  {tag:4s} = {desc}')
print()
print('-' * 70)

for sent in test_sents:
    jieba_result = segment_with_pos(sent)
    hmm_result = tag_raw_sentence(sent, decoder)

    jieba_str = ' '.join(f'{w}/{t}' for w, t in jieba_result)
    hmm_str = ' '.join(f'{w}/{t}' for w, t in hmm_result)

    print(f'原文:     {sent}')
    print(f'jieba:    {jieba_str}')
    print(f'HMM:      {hmm_str}')
    print()

jieba 标注 vs HMM 标注对比:

词性标签含义:
  n    = 名词
  v    = 动词
  a    = 形容词
  d    = 副词
  r    = 代词
  t    = 时间词
  l    = 习用语
  u    = 助词
  ns   = 地名
  nt   = 机构名
  vn   = 名动词

----------------------------------------------------------------------


原文:     他正在研究自然语言处理技术
jieba:    他/r 正在/t 研究/vn 自然语言/l 处理/v 技术/n
HMM:      他/r 正在/d 研究/v 自然语言/u 处理/vn 技术/n

原文:     北京大学是中国最著名的大学之一
jieba:    北京大学/nt 是/v 中国/ns 最/d 著名/a 的/uj 大学/n 之一/r
HMM:      北京大学/nt 是/v 中国/ns 最/d 著名/a 的/u 大学/n 之一/r

原文:     国务院总理主持召开了常务会议
jieba:    国务院/nt 总理/n 主持/v 召开/v 了/ul 常务会议/n
HMM:      国务院/nt 总理/n 主持/v 召开/v 了/u 常务会议/n



### 5.3 自由输入测试

In [14]:
# 自由输入测试
my_input = '2024年中国在人工智能领域取得了重大突破'

print(f'输入句子: {my_input}')
print()

# Step 1: 分词
words = segment(my_input)
print(f'Step 1 - jieba分词: {"  ".join(words)}')
print()

# Step 2: HMM 标注
tagged = tag_raw_sentence(my_input, decoder)
print(f'Step 2 - HMM标注:')
for word, tag in tagged:
    print(f'  {word:8s} → {tag}')
print()

# 完整结果
result_str = tag_raw_sentence_str(my_input, decoder)
print(f'完整结果: {result_str}')

输入句子: 2024年中国在人工智能领域取得了重大突破

Step 1 - jieba分词: 2024  年  中国  在  人工智能  领域  取得  了  重大突破

Step 2 - HMM标注:
  2024     → m
  年        → q
  中国       → ns
  在        → p
  人工智能     → n
  领域       → n
  取得       → v
  了        → u
  重大突破     → n

完整结果: 2024/m 年/q 中国/ns 在/p 人工智能/n 领域/n 取得/v 了/u 重大突破/n
